# E7 — C2 generations on the GPU (generation only; nothing is scored here)

Produces the **10 generations** the canonical gate v2 needs for C2 — 2 classes x
G=5 x n_main 8,000 = **80,000 URLs** — from E7 in **unconditional** mode, through
the gate's **own** code path and parameters. Each file is named exactly as the gate
on the laptop will look it up, so the local run finds them in `--gen-cache` and
only scores. Scoring stays local: the instrument must be the same one that
scored E5.

**Only the forward pass runs on the GPU.** `generate()` moves the logits to the
CPU and samples there with a seeded CPU generator, so these generations should
match what the laptop would produce almost exactly. The local check
`research/e7/c2_gpu_verify.py` measures that before anything is scored.

**Attach three inputs** (right panel → *Add Input*):

| input | what | found by |
|---|---|---|
| `uie-corpus` | the data bundle (only its suffix table is used) | `bundle.manifest.json` |
| `uie-e7-code` | the E7 code | `E7_CODE_MANIFEST.json` |
| **your E7 training notebook's output** (*Your Work → Notebooks*) | `best.pt` | its SHA-256 |

**Settings:** *Accelerator* → **GPU T4 x2** (or P100). *Internet* → **On**
(cell 2 installs `tldextract`). **Run All.** Expect about an hour: sampling is
per-row Python on the CPU even here.

In [ ]:
# --- 1. locate the three inputs ------------------------------------------------
import hashlib
from pathlib import Path

E7_SHA256 = "916d03857d7d3db4101d1b106293a83905f4b317676ee7e144b018170cb36065"  # best_e7.pt


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


root = Path("/kaggle/input")
data_hits = sorted(root.rglob("bundle.manifest.json")) if root.exists() else []
code_hits = sorted(root.rglob("E7_CODE_MANIFEST.json")) if root.exists() else []
pts = {p: sha256(p) for p in sorted(root.rglob("*.pt"))} if root.exists() else {}
ckpt_hits = [p for p, h in pts.items() if h == E7_SHA256]

print("checkpoints seen:")
for p, h in pts.items():
    print(f"   {p}  sha256 {h[:16]}")
if not data_hits or not code_hits or not ckpt_hits:
    raise SystemExit(
        "Attach all three inputs (right panel -> Add Input):\n"
        "  uie-corpus   -- has bundle.manifest.json\n"
        "  uie-e7-code  -- has E7_CODE_MANIFEST.json\n"
        "  the E7 training notebook's OUTPUT -- its best.pt must have sha256 "
        f"{E7_SHA256[:16]}... (final.pt is NOT the one)\n"
        "Inputs mount when the session starts, so re-run after attaching.")

DATA, CODE, CKPT = data_hits[0].parent, code_hits[0].parent, ckpt_hits[0]
print("data:", DATA)
print("code:", CODE)
print("ckpt:", CKPT, "(sha256 verified: best_e7)")

In [ ]:
# --- 2. environment: install first, then run the E7 code and PROVE it --------
import importlib, json, subprocess, sys
from importlib import metadata

# tldextract FIRST: not in the Kaggle image, and `uie` imports it on load.
# Pinned to the version the local gate parses with.
TLDEXTRACT = "5.3.1"


def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


if installed("tldextract") != TLDEXTRACT:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"tldextract=={TLDEXTRACT}"], check=True)
    except subprocess.CalledProcessError:
        raise SystemExit("pip could not install tldextract: "
                         "Settings -> Internet -> On, then rerun.")
    importlib.invalidate_caches()
assert installed("tldextract") == TLDEXTRACT, installed("tldextract")
print("tldextract:", TLDEXTRACT)

# CODE ONLY on the path: the data bundle carries an older `uie`.
sys.path.insert(0, str(CODE))
for name in [m for m in sys.modules if m == "uie" or m.startswith("uie.")]:
    del sys.modules[name]

import uie
manifest = json.loads((CODE / "E7_CODE_MANIFEST.json").read_text())
print("uie imported from:", uie.__file__)
assert str(CODE) in uie.__file__, "the E7 code is SHADOWED -- check sys.path"
bad = [f for f, h in manifest["files"].items()
       if hashlib.sha256((CODE / f).read_bytes()).hexdigest() != h]
print(f"code integrity: {len(manifest['files']) - len(bad)}/{len(manifest['files'])} "
      f"files match the manifest")
assert not bad, f"files differ from the manifest: {bad}"
print(f"E7 code: commit {manifest['commit'][:12]} on {manifest['branch']}")

import torch
print("torch:", torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then rerun.")
print("gpu  :", torch.cuda.get_device_name(0))

In [ ]:
# --- 3. the gate's parameters, and the inputs they must match -----------------
# Every value is the canonical gate's (uie/corpus/run_eval.py, gate v2 baseline).
from uie.corpus.run_eval import _file_sha12, gen_cache_path
from uie.suffixes import SuffixTable, generate_with_suffixes
from uie.train import load_checkpoint

SEED = 20260806                        # run_eval.main default
N_MAIN = 8000
G = 5
DECODER = "temperature"
DECODER_KWARGS = {"temperature": 1.0}  # what `--temperature 1.0` passes
DELTA = {"benign": 0.22, "phishing": 0.30}
# The gate keys its cache on the checkpoint's FILE STEM + sha. Here the file is
# `best.pt`; on the laptop it is `data/best_e7.pt`. Name for the laptop.
LOCAL_STEM = "best_e7"
TABLE_SHA256 = "aaa6fb853ea538d41ba4c10661b89faad2a687b869208772c096c7e507ee866d"
OUT = Path("/kaggle/working/gen_cache_e7_gpu")

table_path = DATA / "suffix_table.json"
assert sha256(table_path) == TABLE_SHA256, (
    "suffix_table.json differs from the laptop's data/bundle2 copy -- STOP: the "
    "suffix draws would not be the gate's")
suffixes = SuffixTable.load(table_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, tokenizer, _ = load_checkpoint(CKPT, device=device)
# load_checkpoint builds the model on the CPU; the gate moves it on the very
# next line (run_eval.py: `model.to(device).eval()`). Attempt 1 of this notebook
# omitted that line and died on the first forward pass. Check the PARAMETERS'
# device, not the variable's name for it.
model.to(device).eval()
where = next(model.parameters()).device
assert where.type == device.type, f"model is on {where}, inputs go to {device}"
assert getattr(tokenizer, "include_fields", False), "not an E7 checkpoint"
assert tokenizer.vocab_size == 107, tokenizer.vocab_size
ckpt_tag = f"{LOCAL_STEM}-{_file_sha12(CKPT)}"
print("suffix table: sha256 verified (same as the laptop's)")
print("model parameters on", where, "| vocab", tokenizer.vocab_size, "| cache tag", ckpt_tag)
print("unconditional mode: generate() receives no `fields`, so it appends <P:?> <Q:?>")

In [ ]:
# --- 4. generate: the gate's exact loop ----------------------------------------
import time

OUT.mkdir(parents=True, exist_ok=True)
jobs = []
for label in tokenizer.classes:                     # benign, phishing -- gate order
    d = float(DELTA.get(label, 0.0))
    fb = {"PTH": {tokenizer.eos_id: -d}} if d else None
    for g in range(G):
        gen_seed = SEED + 1 + g * 1000              # the gate's generation seeds
        jobs.append((label, g, d, fb, gen_seed,
                     gen_cache_path(OUT, ckpt_tag, label, N_MAIN, DECODER,
                                    DECODER_KWARGS, d, gen_seed)))

started, records = time.time(), []
for k, (label, g, d, fb, gen_seed, path) in enumerate(jobs, 1):
    t0 = time.time()
    if path.is_file():
        urls = json.loads(path.read_text(encoding="utf-8"))
        print(f"[{k}/{len(jobs)}] {label} g{g}: already here, kept")
    else:
        urls = generate_with_suffixes(model, tokenizer, label=label, n=N_MAIN,
                                      table=suffixes, decoder=DECODER, seed=gen_seed,
                                      device=device, field_bias=fb,
                                      decoder_kwargs=DECODER_KWARGS or None)
        path.write_text(json.dumps(urls), encoding="utf-8")
    took = time.time() - t0
    records.append({"file": path.name, "label": label, "generation": g,
                    "gen_seed": gen_seed, "delta": d, "n": len(urls),
                    "sha256": sha256(path), "seconds": round(took, 1)})
    left = (time.time() - started) / k * (len(jobs) - k)
    print(f"[{k}/{len(jobs)}] {label} g{g} seed {gen_seed}: {len(urls)} urls "
          f"in {took / 60:.1f} min  (about {left / 60:.0f} min left)", flush=True)

In [ ]:
# --- 5. provenance, and a look at what was made -------------------------------
import platform

prov = {
    "purpose": "E7 C2 generations (PREREG_E7.md section 4, C2), generation only",
    "gate_parameters": {"seed": SEED, "n_main": N_MAIN, "generations": G,
                        "decoder": DECODER, "decoder_kwargs": DECODER_KWARGS,
                        "delta": DELTA, "frame": "domain", "mode": "unconditional",
                        "path": "uie.suffixes.generate_with_suffixes"},
    "device": {"forward": str(device), "gpu": torch.cuda.get_device_name(0),
               "sampling": "cpu (seeded torch.Generator, as in generate())"},
    "environment": {"torch": torch.__version__, "python": platform.python_version(),
                    "tldextract": metadata.version("tldextract")},
    "code_commit": manifest["commit"],
    "checkpoint": {"source": str(CKPT), "sha256": sha256(CKPT), "local_name": "data/best_e7.pt"},
    "suffix_table_sha256": TABLE_SHA256,
    "files": records,
    "total_minutes": round((time.time() - started) / 60, 1),
}
(OUT / "PROVENANCE.json").write_text(json.dumps(prov, indent=1), encoding="utf-8")

ok = (len(records) == len(tokenizer.classes) * G
      and all(r["n"] == N_MAIN for r in records))
print("files:", len(records), "| every file has", N_MAIN, "urls:", ok)
for label in tokenizer.classes:
    first = json.loads((OUT / next(r["file"] for r in records
                                   if r["label"] == label)).read_text())
    print(f"\n{label} examples:", *first[:4], sep="\n  ")
print("\nDONE" if ok else "\nINCOMPLETE -- do not use")

---

## Download

The *Output* tab holds `gen_cache_e7_gpu/`: 10 generation files plus
`PROVENANCE.json` (about 5 MB). Download it (the zip is fine) into `Downloads`.

**Nothing is read yet.** Locally, `c2_gpu_verify.py` first checks every file name
against the gate's own key, the counts and hashes, and regenerates a sample of
suffix groups on the laptop's CPU to compare byte for byte. Only then does gate
v2 score them — and only if C1 has passed.